# Thesis Defence Analysis Notebook
**Purpose:** Document LGS defence (paper hyperparameters), time PatchBlock stage-by-stage,
create PatchBlock defended dataset (info=0.65), and save all thesis-ready figures.

**LGS Paper Hyperparameters:**
- CARLA-GeAR (C2): sigma=7, threshold=0.10, morph_kernel=3, dilate_iters=0
- Isaac Sim (C3): sigma=5, threshold=0.10, morph_kernel=3, dilate_iters=0


## 1. Setup & Imports

In [ ]:
# Mount Drive (Colab)
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip -q install ultralytics==8.4.7 scikit-learn kornia tqdm

import os, glob, re, yaml, time, shutil
import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import torch
from PIL import Image
from pathlib import Path
from tqdm import tqdm
from sklearn.ensemble import IsolationForest
from ultralytics import YOLO

# ─── Output folder for all thesis figures ───────────────────────────────────
THESIS_FIGS = Path("/content/thesis_figures")
THESIS_FIGS.mkdir(exist_ok=True)
print("Thesis figures will be saved to:", THESIS_FIGS)

# ─── Global plot style ───────────────────────────────────────────────────────
plt.rcParams.update({
    'font.size': 11, 'axes.titlesize': 12, 'axes.labelsize': 11,
    'legend.fontsize': 10, 'figure.dpi': 150,
    'savefig.dpi': 300, 'savefig.bbox': 'tight'
})

def save_fig(name):
    path = THESIS_FIGS / f"{name}.png"
    plt.savefig(path, bbox_inches='tight', dpi=300)
    print(f"  ✅ Saved: {path}")

print("Setup complete.")

## 2. Paths & Model — EDIT THESE

In [ ]:
# ─── EDIT THESE PATHS ────────────────────────────────────────────────────────
MODEL_PATH      = "/content/drive/MyDrive/yolo_trained_models/2dod_checkpoints/best.pt"
PATCHED_ROOT    = "/content/drive/MyDrive/Patched Datasets/patched_dataset1"  # one scenario for demo
OUT_ROOT        = "/content/drive/MyDrive/Patched Datasets/Defended"

# Pick ONE patched image for visualisation cells
DEMO_IMG_PATH   = "/content/drive/MyDrive/Patched Datasets/patched_dataset1/images/test/rgb_0001.png"

# ─── Paper settings ─────────────────────────────────────────────────────────
IMGSZ     = 640
NMS_IOU   = 0.6
IOU_MATCH = 0.50
TAU_LOAD  = 0.001
TAUS      = [0.001, 0.3, 0.5]
OVERLOAD_K = 50

# ─── LGS paper hyperparameters ──────────────────────────────────────────────
LGS_SIGMA_CARLA  = 7.0    # C2 — CARLA-GeAR (held-out scenarios)
LGS_THR_CARLA    = 0.10
LGS_MORPH_CARLA  = 3
LGS_DIL_CARLA    = 0

LGS_SIGMA_ISAAC  = 5.0    # C3 — Isaac Sim (domain re-tuned)
LGS_THR_ISAAC    = 0.10
LGS_MORPH_ISAAC  = 3
LGS_DIL_ISAAC    = 0

# ─── PatchBlock hyperparameters ──────────────────────────────────────────────
PB_K            = 50
PB_STRIDE       = 10
PB_OUTLIER_FRAC = 0.02
PB_BINS         = 64
PB_INFO         = 0.65   # SVD retention threshold (thesis version)

# ─── Model ───────────────────────────────────────────────────────────────────
model  = YOLO(MODEL_PATH)
device = next(model.model.parameters()).device
model.model.eval()
print("Model loaded. Device:", device)

## 3. Defence Functions

In [ ]:
# ─── LGS (paper-identical) ───────────────────────────────────────────────────
def lgs_defense(img_rgb_uint8, sigma=7.0, threshold=0.10,
                morph_kernel_size=3, dilate_iters=0, soft_mask=True):
    """Local Gradient Smoothing — paper hyperparameters by default (C2)."""
    img = img_rgb_uint8.astype(np.float32) / 255.0
    gray = cv2.cvtColor((img * 255).astype(np.uint8), cv2.COLOR_RGB2GRAY)

    hf = np.abs(cv2.Laplacian(gray, cv2.CV_32F, ksize=3))
    hf_norm = hf / (hf.max() + 1e-8)

    mask = (hf_norm > threshold).astype(np.uint8)

    if morph_kernel_size > 0:
        k = np.ones((morph_kernel_size, morph_kernel_size), np.uint8)
        mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN,  k)
        mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, k)

    if dilate_iters > 0:
        kd = np.ones((morph_kernel_size, morph_kernel_size), np.uint8)
        mask = cv2.dilate(mask, kd, iterations=dilate_iters)

    mask_f = mask.astype(np.float32)
    if soft_mask:
        mask_f = cv2.GaussianBlur(mask_f, (0, 0), 1.0)
        mask_f = np.clip(mask_f, 0, 1)

    blurred = cv2.GaussianBlur(img, (0, 0), sigmaX=sigma, sigmaY=sigma)
    out = img * (1 - mask_f[..., None]) + blurred * mask_f[..., None]
    out = np.clip(out, 0, 1)
    return (out * 255).astype(np.uint8), hf_norm, mask_f


# ─── PatchBlock helpers ───────────────────────────────────────────────────────
def mi_hist2d(a, b, bins=64, eps=1e-12):
    h2d, _, _ = np.histogram2d(a.ravel(), b.ravel(), bins=bins, range=[[0,256],[0,256]])
    pxy = h2d / (h2d.sum() + eps)
    px = pxy.sum(axis=1, keepdims=True)
    py = pxy.sum(axis=0, keepdims=True)
    denom = px @ py
    nz = pxy > 0
    return float((pxy[nz] * np.log((pxy[nz] + eps) / (denom[nz] + eps))).sum())

def entropy_uint8(a, bins=64, eps=1e-12):
    h, _ = np.histogram(a.ravel(), bins=bins, range=(0, 256))
    p = h.astype(np.float32) / (h.sum() + eps)
    p = p[p > 0]
    return float(-(p * np.log(p + eps)).sum())

def hf_energy(a):
    return float(cv2.Laplacian(a, cv2.CV_32F, ksize=3).var())

def pb_chunk_features(img_rgb, k=50, stride=10, bins=64):
    H, W = img_rgb.shape[:2]
    hsv  = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2HSV)
    S    = hsv[:, :, 1]
    gray = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2GRAY)
    coords, feats = [], []
    for y in range(0, H - k + 1, stride):
        for x in range(0, W - k + 1, stride):
            p = img_rgb[y:y+k, x:x+k]
            R, G, B = p[:,:,0], p[:,:,1], p[:,:,2]
            mi_rgb = mi_hist2d(R, G, bins) + mi_hist2d(G, B, bins) + mi_hist2d(R, B, bins)
            P = gray[y:y+k, x:x+k]
            nbr = []
            for dy, dx in [(-stride,0),(stride,0),(0,-stride),(0,stride)]:
                y2, x2 = y+dy, x+dx
                if 0 <= y2 <= H-k and 0 <= x2 <= W-k:
                    nbr.append(mi_hist2d(P, gray[y2:y2+k, x2:x2+k], bins))
            mi_local = float(np.mean(nbr)) if nbr else 0.0
            Sp = S[y:y+k, x:x+k]
            feats.append([mi_rgb, mi_local, float(Sp.mean())/255.0, entropy_uint8(Sp, bins), hf_energy(Sp)])
            coords.append((y, x))
    return coords, np.array(feats, dtype=np.float32)

def pb_separate(coords, feats, img_hw, k=50, stride=10, outlier_frac=0.02):
    H, W = img_hw
    n = len(coords)
    if n == 0: return np.array([], dtype=int), None
    iso = IsolationForest(n_estimators=200, max_samples=max(2, int(0.3*n)),
                          contamination=outlier_frac, random_state=0, n_jobs=-1)
    iso.fit(feats)
    scores = -iso.decision_function(feats)
    r = max(1, int(outlier_frac * n))
    top = np.argsort(scores)[-r:]
    gh = (H - k) // stride + 1
    gw = (W - k) // stride + 1
    mask = np.zeros((gh, gw), np.uint8)
    for i in top:
        y, x = coords[i]
        mask[y // stride, x // stride] = 1
    num, labels = cv2.connectedComponents(mask)
    if num <= 1: return top.astype(int), scores
    best_lab, best_score = 1, -1e18
    for lab in range(1, num):
        idxs = [i for i in top if labels[coords[i][0]//stride, coords[i][1]//stride] == lab]
        if not idxs: continue
        f = feats[idxs]
        s = f[:,2].mean()*3.0 + f[:,4].mean()*0.002 + f[:,3].mean()*1.0
        if s > best_score: best_score, best_lab = s, lab
    keep = [i for i in top if labels[coords[i][0]//stride, coords[i][1]//stride] == best_lab]
    return np.array(keep, dtype=int), scores

def svd_mitigate(patch_float, info=0.65):
    k = patch_float.shape[0]
    M = patch_float.reshape(k, -1)
    U, S, Vt = np.linalg.svd(M, full_matrices=False)
    cum = np.cumsum(S) / (S.sum() + 1e-12)
    r = int(np.searchsorted(cum, info) + 1)
    return ((U[:,:r] * S[:r]) @ Vt[:r,:]).reshape(k, k, 3).astype(np.float32)

def pb_mitigate(img_rgb, coords, keep_idx, k=50, info=0.65):
    out = img_rgb.astype(np.float32).copy()
    for i in keep_idx:
        y, x = coords[i]
        out[y:y+k, x:x+k] = svd_mitigate(out[y:y+k, x:x+k], info=info)
    return np.clip(out, 0, 255).astype(np.uint8)

def defend_patchblock(img_rgb, k=PB_K, stride=PB_STRIDE, outlier_frac=PB_OUTLIER_FRAC,
                       bins=PB_BINS, info=PB_INFO):
    """Full PatchBlock pipeline. Returns (defended_img, coords, keep_idx)."""
    coords, feats = pb_chunk_features(img_rgb, k, stride, bins)
    H, W = img_rgb.shape[:2]
    keep_idx, _ = pb_separate(coords, feats, (H, W), k, stride, outlier_frac)
    defended     = pb_mitigate(img_rgb, coords, keep_idx, k, info)
    return defended, coords, keep_idx

print("Defence functions defined.")

## 4. LGS — Thesis Pipeline Figure (4-panel)

In [ ]:
patched = np.array(Image.open(DEMO_IMG_PATH).convert("RGB"))

defended_lgs, hf_norm, mask_f = lgs_defense(
    patched,
    sigma=LGS_SIGMA_CARLA, threshold=LGS_THR_CARLA,
    morph_kernel_size=LGS_MORPH_CARLA, dilate_iters=LGS_DIL_CARLA
)

fig, axes = plt.subplots(1, 4, figsize=(18, 4))
axes[0].imshow(patched);                      axes[0].set_title("Original (patched)")
axes[1].imshow(hf_norm, cmap="hot");          axes[1].set_title("HF map (Laplacian)")
axes[2].imshow(mask_f, cmap="gray");          axes[2].set_title("Mask")
axes[3].imshow(defended_lgs);                 axes[3].set_title("LGS output")
for ax in axes: ax.axis("off")
plt.suptitle(f"LGS Defence Pipeline  (σ={LGS_SIGMA_CARLA}, thr={LGS_THR_CARLA})",
             fontsize=13, y=1.01)
plt.tight_layout()
save_fig("lgs_pipeline_4panel")
plt.show()

# Also save individual panels for flexibility in the thesis
for img, name, cmap in [
    (patched,      "lgs_panel_original",  None),
    (hf_norm,      "lgs_panel_hfmap",     "hot"),
    (mask_f,       "lgs_panel_mask",      "gray"),
    (defended_lgs, "lgs_panel_output",    None),
]:
    fig2, ax2 = plt.subplots(figsize=(5, 4))
    ax2.imshow(img, cmap=cmap); ax2.axis("off")
    plt.tight_layout(pad=0)
    save_fig(name)
    plt.close()

print("\nLGS Parameters used (CARLA-GeAR C2):")
print(f"  sigma={LGS_SIGMA_CARLA}, threshold={LGS_THR_CARLA}, morph_kernel={LGS_MORPH_CARLA}, dilate_iters={LGS_DIL_CARLA}")

## 5. LGS — Clean vs Patched vs Defended Comparison Figure

In [ ]:
# You need a clean image path for the same frame
CLEAN_IMG_PATH = DEMO_IMG_PATH.replace("patched_dataset1", "Clean/clean_dataset1")

if os.path.exists(CLEAN_IMG_PATH):
    clean = np.array(Image.open(CLEAN_IMG_PATH).convert("RGB"))

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    axes[0].imshow(clean);        axes[0].set_title("Clean")
    axes[1].imshow(patched);      axes[1].set_title("Patched")
    axes[2].imshow(defended_lgs); axes[2].set_title("LGS Defended")
    for ax in axes: ax.axis("off")
    plt.suptitle("Clean / Patched / LGS Defended", fontsize=13, y=1.01)
    plt.tight_layout()
    save_fig("lgs_clean_patched_defended")
    plt.show()
else:
    print(f"Clean image not found at: {CLEAN_IMG_PATH}")
    print("Adjust CLEAN_IMG_PATH manually.")

## 6. PatchBlock — Stage-by-Stage Timing Analysis

In [ ]:
N_TIMING_RUNS = 10  # repeat for stable estimates

timing_results = {"Stage 1: Chunking": [], "Stage 2: Separating": [], "Stage 3: Mitigating": [], "Total": []}

for run in range(N_TIMING_RUNS):
    t0 = time.perf_counter()
    coords, feats = pb_chunk_features(patched, PB_K, PB_STRIDE, PB_BINS)
    t1 = time.perf_counter()
    H, W = patched.shape[:2]
    keep_idx, _ = pb_separate(coords, feats, (H, W), PB_K, PB_STRIDE, PB_OUTLIER_FRAC)
    t2 = time.perf_counter()
    defended_pb = pb_mitigate(patched, coords, keep_idx, PB_K, PB_INFO)
    t3 = time.perf_counter()

    timing_results["Stage 1: Chunking"].append((t1-t0)*1000)
    timing_results["Stage 2: Separating"].append((t2-t1)*1000)
    timing_results["Stage 3: Mitigating"].append((t3-t2)*1000)
    timing_results["Total"].append((t3-t0)*1000)

print("=" * 55)
print(f"{'PatchBlock Stage Timing':<35} {'Mean (ms)':>10} {'Std (ms)':>10}")
print("=" * 55)
timing_df_rows = []
for stage, times in timing_results.items():
    mean_t = np.mean(times)
    std_t  = np.std(times)
    pct    = mean_t / np.mean(timing_results["Total"]) * 100
    print(f"{stage:<35} {mean_t:>10.1f} {std_t:>10.1f}  ({pct:.1f}%)")
    timing_df_rows.append({"Stage": stage, "Mean (ms)": round(mean_t,1), "Std (ms)": round(std_t,1), "% of Total": round(pct,1)})
print("=" * 55)

timing_df = pd.DataFrame(timing_df_rows)
print("\nFull timing table:")
print(timing_df.to_string(index=False))

# Save timing table as CSV
timing_df.to_csv(THESIS_FIGS / "patchblock_stage_timing.csv", index=False)
print("\n✅ Timing CSV saved.")

# Bar chart
fig, ax = plt.subplots(figsize=(8, 4))
stage_labels = [r.split(":")[1].strip() for r in list(timing_results.keys())[:3]]
means = [np.mean(timing_results[k]) for k in list(timing_results.keys())[:3]]
stds  = [np.std(timing_results[k])  for k in list(timing_results.keys())[:3]]
bars = ax.bar(stage_labels, means, yerr=stds, capsize=5, color=['#4C72B0','#DD8452','#55A868'])
ax.set_ylabel("Time (ms)")
ax.set_title(f"PatchBlock Stage Latency (n={N_TIMING_RUNS} runs, image {patched.shape[1]}×{patched.shape[0]})")
for bar, m in zip(bars, means):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + max(stds)*0.1,
            f"{m:.0f} ms", ha='center', va='bottom', fontsize=10)
plt.tight_layout()
save_fig("patchblock_stage_timing_bar")
plt.show()

print(f"\n⚠️  Bottleneck: Stage 2 (Isolation Forest) accounts for the largest share of total time.")
print(f"    This makes PatchBlock unsuitable for real-time edge deployment.")

## 7. PatchBlock — Visual Pipeline Figure (Thesis-Ready)

In [ ]:
# Run PatchBlock on demo image
defended_pb, coords, keep_idx = defend_patchblock(patched)

# Draw selected anomalous chunk boxes
boxed = patched.copy()
for i in keep_idx:
    y, x = coords[i]
    cv2.rectangle(boxed, (x, y), (x+PB_K, y+PB_K), (255, 0, 0), 2)

print(f"PatchBlock: {len(keep_idx)} anomalous chunks selected")
print(f"Info parameter: {PB_INFO} (SVD retention threshold)")

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
axes[0].imshow(patched);    axes[0].set_title("Patched Input")
axes[1].imshow(boxed);      axes[1].set_title(f"Stage 2: Anomalous Chunks (n={len(keep_idx)})")
axes[2].imshow(defended_pb); axes[2].set_title(f"Stage 3: SVD Mitigated (info={PB_INFO})")
for ax in axes: ax.axis("off")
plt.suptitle(f"PatchBlock Defence Pipeline  (K={PB_K}, stride={PB_STRIDE}, outlier_frac={PB_OUTLIER_FRAC}, info={PB_INFO})",
             fontsize=12, y=1.01)
plt.tight_layout()
save_fig("patchblock_pipeline_3panel")
plt.show()

# Save individual panels
for img, name in [(patched, "pb_panel_input"), (boxed, "pb_panel_chunks"), (defended_pb, "pb_panel_output")]:
    fig2, ax2 = plt.subplots(figsize=(5, 4))
    ax2.imshow(img); ax2.axis("off")
    plt.tight_layout(pad=0)
    save_fig(name)
    plt.close()

## 8. LGS vs PatchBlock Side-by-Side Comparison

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(22, 5))
axes[0].imshow(patched);       axes[0].set_title("Patched (no defence)")
axes[1].imshow(defended_lgs);  axes[1].set_title(f"LGS  (σ={LGS_SIGMA_CARLA}, thr={LGS_THR_CARLA})")
axes[2].imshow(defended_pb);   axes[2].set_title(f"PatchBlock SVD  (info={PB_INFO})")

# Difference maps
diff_lgs = np.abs(patched.astype(int) - defended_lgs.astype(int)).mean(axis=2).astype(np.uint8)
diff_pb  = np.abs(patched.astype(int) - defended_pb.astype(int)).mean(axis=2).astype(np.uint8)
diff_both = np.stack([diff_lgs, diff_pb, np.zeros_like(diff_lgs)], axis=2)
axes[3].imshow(diff_both);    axes[3].set_title("Pixel Δ: LGS (red) / PatchBlock (green)")

for ax in axes: ax.axis("off")
plt.suptitle("Defence Comparison", fontsize=13, y=1.01)
plt.tight_layout()
save_fig("defence_comparison_lgs_vs_patchblock")
plt.show()

# Print pixel-level stats
print(f"Mean absolute pixel change:")
print(f"  LGS:        {diff_lgs.mean():.2f}")
print(f"  PatchBlock: {diff_pb.mean():.2f}")

## 9. Build PatchBlock Defended Dataset (info=0.65)

In [ ]:
# Build a PatchBlock-defended copy of PATCHED_ROOT with info=0.65
PB_OUT = Path(OUT_ROOT) / f"patched_dataset1_defended_PatchBlock_SVD_info{int(PB_INFO*100):02d}"
PB_OUT.mkdir(parents=True, exist_ok=True)

src_root  = Path(PATCHED_ROOT)
img_dir   = src_root / "images" / "test"
lab_dir   = src_root / "labels" / "test"
out_img   = PB_OUT / "images" / "test"
out_lab   = PB_OUT / "labels" / "test"
out_img.mkdir(parents=True, exist_ok=True)
out_lab.mkdir(parents=True, exist_ok=True)

img_paths = sorted(list(img_dir.glob("*.png")) + list(img_dir.glob("*.jpg")))
print(f"Building PatchBlock defended dataset (info={PB_INFO}) — {len(img_paths)} images")

latencies = []
for p in tqdm(img_paths):
    img = np.array(Image.open(p).convert("RGB"))

    t0 = time.perf_counter()
    defended, _, _ = defend_patchblock(img, info=PB_INFO)
    latencies.append((time.perf_counter() - t0) * 1000)

    Image.fromarray(defended).save(out_img / p.name)

    # Copy labels unchanged
    lab_src = lab_dir / (p.stem + ".txt")
    if lab_src.exists():
        shutil.copy(lab_src, out_lab / lab_src.name)

# Copy data.yaml if present
yaml_src = src_root / "data.yaml"
if yaml_src.exists():
    shutil.copy(yaml_src, PB_OUT / "data.yaml")

print(f"\n✅ Dataset saved to: {PB_OUT}")
print(f"\nPer-image PatchBlock latency (ms):")
print(f"  Mean: {np.mean(latencies):.1f}")
print(f"  Std:  {np.std(latencies):.1f}")
print(f"  Min:  {np.min(latencies):.1f}")
print(f"  Max:  {np.max(latencies):.1f}")
print(f"  p95:  {np.percentile(latencies, 95):.1f}")
print(f"\n⚠️  At ~{np.mean(latencies):.0f} ms/image, PatchBlock cannot run in real time on edge hardware.")

In [ ]:
import time
import numpy as np
from PIL import Image
from sklearn.ensemble import IsolationForest
import cv2

# Load one image
img = np.array(Image.open(img_paths[0]).convert('RGB'))
H, W = img.shape[:2]

def count_chunks(k, stride):
    n = 0
    for y in range(0, H - k + 1, stride):
        for x in range(0, W - k + 1, stride):
            n += 1
    return n

def time_patchblock(k, stride, n_runs=5):
    times = {'chunking': [], 'isolation_forest': [], 'svd': [], 'total': []}

    for _ in range(n_runs):
        # Stage 1: Chunking
        t0 = time.perf_counter()
        coords, feats = pb_chunk_features(img, k=k, stride=stride, bins=PB_BINS)
        t1 = time.perf_counter()

        # Stage 2: Isolation Forest
        iso = IsolationForest(
            n_estimators=200,
            max_samples=max(2, int(0.3 * len(coords))),
            contamination=PB_OUTLIER_FRAC,
            random_state=0, n_jobs=-1
        )
        iso.fit(feats)
        scores = -iso.decision_function(feats)
        r = max(1, int(PB_OUTLIER_FRAC * len(coords)))
        top = np.argsort(scores)[-r:]
        t2 = time.perf_counter()

        # Stage 3: SVD mitigate
        defended = pb_mitigate(img, coords, top, k=k, info=PB_INFO)
        t3 = time.perf_counter()

        times['chunking'].append((t1-t0)*1000)
        times['isolation_forest'].append((t2-t1)*1000)
        times['svd'].append((t3-t2)*1000)
        times['total'].append((t3-t0)*1000)

    n_chunks = count_chunks(k, stride)
    return {
        'k': k, 'stride': stride, 'n_chunks': n_chunks,
        'chunking_ms':   round(np.mean(times['chunking']), 1),
        'iso_forest_ms': round(np.mean(times['isolation_forest']), 1),
        'svd_ms':        round(np.mean(times['svd']), 1),
        'total_ms':      round(np.mean(times['total']), 1),
    }

# ── Sweep ────────────────────────────────────────────────────────────────────
configs = [
    # (k,  stride)  — original is k=50, stride=10
    (50,  10),   # original: ~3000+ chunks
    (50,  20),   # halved resolution
    (50,  30),   # 3× sparser
    (50,  50),   # no overlap at all
    (80,  40),   # larger chunks, sparser
    (100, 50),   # even larger
]

print(f"{'k':>4} {'stride':>6} {'chunks':>7} {'chunk(ms)':>10} {'IF(ms)':>8} {'SVD(ms)':>8} {'total(ms)':>10}")
print('-' * 65)
results = []
for k, stride in configs:
    r = time_patchblock(k, stride)
    results.append(r)
    print(f"{r['k']:>4} {r['stride']:>6} {r['n_chunks']:>7} "
          f"{r['chunking_ms']:>10.1f} {r['iso_forest_ms']:>8.1f} "
          f"{r['svd_ms']:>8.1f} {r['total_ms']:>10.1f}")

print()
print("Original (k=50, stride=10):", results[0]['total_ms'], 'ms')
best = min(results, key=lambda x: x['total_ms'])
print(f"Fastest config: k={best['k']}, stride={best['stride']} → {best['total_ms']} ms")
print(f"Speedup vs original: {results[0]['total_ms']/best['total_ms']:.1f}×")

In [ ]:
%%writefile /content/evaluation_ab.py
import argparse, json, re, subprocess, sys
from pathlib import Path
import numpy as np
from ultralytics import YOLO

IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

# ----------------- utils -----------------
def run(cmd, log_path: Path):
    log_path.parent.mkdir(parents=True, exist_ok=True)
    with log_path.open("w", encoding="utf-8") as f:
        p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
        for line in p.stdout:
            sys.stdout.write(line)
            f.write(line)
        rc = p.wait()
    return rc

def percentile(arr, p):
    if arr is None or len(arr) == 0:
        return None
    return float(np.percentile(np.asarray(arr, dtype=np.float32), p))

def mean_std(values):
    vals = [v for v in values if isinstance(v, (int, float)) and not np.isnan(v)]
    if not vals:
        return {"mean": None, "std": None, "n": 0}
    a = np.asarray(vals, dtype=np.float32)
    return {"mean": float(a.mean()), "std": float(a.std(ddof=1)) if len(a) > 1 else 0.0, "n": int(len(a))}

def safe_mkdir(p: Path):
    p.mkdir(parents=True, exist_ok=True)
    return p

def model_names_as_list(model: YOLO):
    names = getattr(model, "names", None)
    if names is None:
        raise ValueError("Model has no .names; cannot build dataset YAML (needs names/nc).")

    if isinstance(names, dict):
        max_k = max(int(k) for k in names.keys())
        out = [None] * (max_k + 1)
        for k, v in names.items():
            out[int(k)] = str(v)
        for i, v in enumerate(out):
            if v is None:
                out[i] = f"class_{i}"
        return out

    if isinstance(names, (list, tuple)):
        return [str(x) for x in names]

    raise ValueError(f"Unsupported model.names type: {type(names)}")

def write_yaml(out_yaml: Path, root: Path, split: str, scenario: str, names):
    """
    Dataset layout assumed:
      root/images/<split>/<scenario>
      root/labels/<split>/<scenario>
    We set train/val/test all to the same relative path so yolo val can use split=<split>.
    """
    out_yaml.parent.mkdir(parents=True, exist_ok=True)
    rel = f"images/{split}/{scenario}"
    content = (
        f"path: {root}\n"
        f"train: {rel}\n"
        f"val: {rel}\n"
        f"test: {rel}\n"
        f"\nnc: {len(names)}\n"
        f"names: {json.dumps(names)}\n"
    )
    out_yaml.write_text(content, encoding="utf-8")

def parse_yolo_val_metrics_from_log(log_path: Path):
    pat_all = re.compile(r"^\s*all\s+(\d+)\s+(\d+)\s+(\d*\.?\d+)\s+(\d*\.?\d+)\s+(\d*\.?\d+)\s+(\d*\.?\d+)\s*$")
    per_class = {}
    pat_cls = re.compile(r"^\s*([A-Za-z0-9 _-]+)\s+(\d+)\s+(\d+)\s+(\d*\.?\d+)\s+(\d*\.?\d+)\s+(\d*\.?\d+)\s+(\d*\.?\d+)\s*$")

    images = instances = None
    P = R = mAP50 = mAP5095 = None

    for ln in log_path.read_text(encoding="utf-8", errors="ignore").splitlines():
        m = pat_all.match(ln)
        if m:
            images = int(m.group(1))
            instances = int(m.group(2))
            P = float(m.group(3))
            R = float(m.group(4))
            mAP50 = float(m.group(5))
            mAP5095 = float(m.group(6))

        mc = pat_cls.match(ln)
        if mc:
            cls = mc.group(1).strip()
            if cls != "all":
                per_class[cls] = {
                    "images": int(mc.group(2)),
                    "instances": int(mc.group(3)),
                    "precision": float(mc.group(4)),
                    "recall": float(mc.group(5)),
                    "mAP50": float(mc.group(6)),
                    "mAP50-95": float(mc.group(7)),
                }

    return {
        "images": images,
        "instances": instances,
        "precision": P,
        "recall": R,
        "mAP50": mAP50,
        "mAP50-95": mAP5095,
        "per_class": per_class,
        "log_path": str(log_path),
    }

def _normalize_device_arg(device_str: str):
    """
    Accepts: 'cpu', '0', '1', '0,1', 'cuda:0', 'mps'
    Ultralytics accepts many of these; for predict() it can be int or str.
    We'll convert pure integers to int for predict() but keep a string for yolo CLI.
    """
    ds = (device_str or "").strip()
    if ds == "":
        return "cpu", "cpu"
    if ds.lower() == "none":
        return "cpu", "cpu"
    # Pure int -> int for predict, string for CLI
    if re.fullmatch(r"\d+", ds):
        return int(ds), ds
    return ds, ds

# ----------------- A) detection density / hallucination-ish -----------------
def compute_detection_density_metrics(model: YOLO, img_dir: Path, imgsz: int, conf_min: float, iou: float, thresholds, det_k_list, device_pred):
    names = model_names_as_list(model)

    det_counts = []
    max_conf_any = []
    has_ge_k = {str(k): 0 for k in det_k_list}
    has_any_conf_ge = {str(t): 0 for t in thresholds}
    total_dets_conf_ge = {str(t): 0 for t in thresholds}
    class_hist = {}

    results_iter = model.predict(
        source=str(img_dir),
        imgsz=imgsz,
        conf=conf_min,
        iou=iou,
        device=device_pred,      # ✅ GPU/CPU here
        stream=True,
        verbose=False,
    )

    n_images = 0
    for r in results_iter:
        n_images += 1
        boxes = r.boxes
        n = 0 if boxes is None else len(boxes)
        det_counts.append(n)

        if n == 0:
            max_conf_any.append(0.0)
        else:
            confs = boxes.conf.detach().cpu().numpy().astype(np.float32)
            max_conf_any.append(float(confs.max()))
            clss = boxes.cls.detach().cpu().numpy().astype(int)

            for c in clss:
                k = names[c] if (0 <= c < len(names)) else str(int(c))
                class_hist[k] = class_hist.get(k, 0) + 1

            for t in thresholds:
                total_dets_conf_ge[str(t)] += int((confs >= t).sum())

        for k in det_k_list:
            if n >= k:
                has_ge_k[str(k)] += 1

        for t in thresholds:
            if max_conf_any[-1] >= t:
                has_any_conf_ge[str(t)] += 1

        if n_images == 1 or (n_images % 10 == 0):
            print(f"[A] {img_dir.name}: processed {n_images} images", flush=True)

    det_counts_np = np.asarray(det_counts, dtype=np.float32)
    max_conf_np = np.asarray(max_conf_any, dtype=np.float32)

    return {
        "images_eval": int(n_images),
        "detections_total_model": int(det_counts_np.sum()),
        "detections_per_image_mean": float(det_counts_np.mean()) if n_images else None,
        "detections_per_image_median": float(np.median(det_counts_np)) if n_images else None,
        "detections_per_image_p95": percentile(det_counts_np, 95),
        "max_conf_any_mean": float(max_conf_np.mean()) if n_images else None,
        "max_conf_any_median": float(np.median(max_conf_np)) if n_images else None,
        "max_conf_any_p95": percentile(max_conf_np, 95),
        "image_rate_dets_ge": {k: (v / n_images if n_images else None) for k, v in has_ge_k.items()},
        "image_rate_any_det_conf_ge": {k: (v / n_images if n_images else None) for k, v in has_any_conf_ge.items()},
        "total_detections_conf_ge": total_dets_conf_ge,
        "predicted_class_histogram": dict(sorted(class_hist.items(), key=lambda kv: kv[1], reverse=True)),
        "config": {
            "img_dir_used": str(img_dir),
            "imgsz": imgsz,
            "conf_min": conf_min,
            "iou": iou,
            "thresholds": thresholds,
            "det_k_list": det_k_list,
            "device": str(device_pred),
        },
    }

# ----------------- B) false positives vs GT -----------------
def read_yolo_gt_labels(txt_path: Path):
    if not txt_path.exists():
        return []
    lines = txt_path.read_text(encoding="utf-8", errors="ignore").strip().splitlines()
    out = []
    for ln in lines:
        parts = ln.strip().split()
        if len(parts) < 5:
            continue
        try:
            c = int(float(parts[0]))
            x, y, w, h = map(float, parts[1:5])
            out.append((c, x, y, w, h))
        except Exception:
            continue
    return out

def xywhn_to_xyxy(x, y, w, h):
    x1 = x - w / 2.0
    y1 = y - h / 2.0
    x2 = x + w / 2.0
    y2 = y + h / 2.0
    return x1, y1, x2, y2

def iou_xyxy(a, b):
    ax1, ay1, ax2, ay2 = a
    bx1, by1, bx2, by2 = b
    ix1 = max(ax1, bx1)
    iy1 = max(ay1, by1)
\small
\renewcommand{\arraystretch}{1.15}
\begin{tabular}{lrrrr}
\toprule
\textbf{Condition} & \textbf{Precision} & \textbf{Recall}
                   & \textbf{mAP50} & \textbf{mAP50--95} \\
\midrule
Clean (reference)  & 0.862 & 0.740 & 0.794 & 0.636 \\
SODA (defended)    & 0.880 & 0.615 & 0.681 & 0.468 \\
\bottomrule
\end{tabular}
\end{table}

\subsubsection{Interpretation}

\paragraph{JPEG.}
Produces negligible change across all metrics in both CARLA-GeAR and
Isaac Sim, confirming it is not an effective defence in this setting.

\paragraph{Gaussian blur and median filter.}
Both reduce raw detection volume but their benefit diminishes at
operational thresholds, with small increases at FP@0.7.

\paragraph{LGS (C3: $\sigma=5$, thr\,=\,0.10).}
With the domain-tuned configuration, LGS provides the strongest
suppression across all operational thresholds (FP@0.3: $-4.20$;
FP@0.5: $-2.68$; FP@0.7: $-1.15$) and is the only smoothing-based
defence that consistently reduces false positives at every tested
confidence level.
    ix2 = min(ax2, bx2)
    iy2 = min(ay2, by2)
    iw = max(0.0, ix2 - ix1)
    ih = max(0.0, iy2 - iy1)
    inter = iw * ih
    area_a = max(0.0, ax2 - ax1) * max(0.0, ay2 - ay1)
    area_b = max(0.0, bx2 - bx1) * max(0.0, by2 - by1)
    union = area_a + area_b - inter
    return inter / union if union > 0 else 0.0

def compute_false_positive_metrics(model: YOLO, img_dir: Path, labels_dir: Path, imgsz: int, conf_min: float, iou: float, thresholds, gt_iou_match: float, device_pred):
    fp_per_image = []
    fp_by_thr = {str(t): [] for t in thresholds}

    total_fp = 0
    total_tp = 0
    total_preds = 0
    total_gt = 0

    results_iter = model.predict(
        source=str(img_dir),
        imgsz=imgsz,
        conf=conf_min,
        iou=iou,
        device=device_pred,     # ✅ GPU/CPU here
        stream=True,
        verbose=False,
    )

    n_images = 0
    for r in results_iter:
        n_images += 1
        img_name = Path(r.path).name
        stem = Path(img_name).stem
        gt_path = labels_dir / f"{stem}.txt"

        gt = read_yolo_gt_labels(gt_path)
        total_gt += len(gt)

        boxes = r.boxes
        if boxes is None or len(boxes) == 0:
            fp_per_image.append(0)
            for t in thresholds:
                fp_by_thr[str(t)].append(0)
            if n_images == 1 or (n_images % 10 == 0):
                print(f"[B] {img_dir.name}: processed {n_images} images", flush=True)
            continue

        pred_cls = boxes.cls.detach().cpu().numpy().astype(int)
        pred_xywhn = boxes.xywhn.detach().cpu().numpy().astype(np.float32)
        pred_conf = boxes.conf.detach().cpu().numpy().astype(np.float32)

        preds = []
        for i in range(len(pred_cls)):
            c = int(pred_cls[i])
            x, y, w, h = pred_xywhn[i].tolist()
            preds.append((c, xywhn_to_xyxy(x, y, w, h), float(pred_conf[i])))

        total_preds += len(preds)

        gt_by_class = {}
        for (c, x, y, w, h) in gt:
            gt_by_class.setdefault(int(c), []).append(xywhn_to_xyxy(x, y, w, h))

        def count_fp_at(thr):
            pf = [(c, box, conf) for (c, box, conf) in preds if conf >= thr]
            pf.sort(key=lambda z: z[2], reverse=True)

            tp = 0
            fp = 0
            remaining = {c: list(lst) for c, lst in gt_by_class.items()}

            for c, pbox, _ in pf:
                best_iou = 0.0
                best_j = None
                gtl = remaining.get(c, [])
                for j, gbox in enumerate(gtl):
                    iouv = iou_xyxy(pbox, gbox)
                    if iouv > best_iou:
                        best_iou = iouv
                        best_j = j
                if best_j is not None and best_iou >= gt_iou_match:
                    tp += 1
                    gtl.pop(best_j)
                    remaining[c] = gtl
                else:
                    fp += 1
            return tp, fp, len(pf)

        tp0, fp0, _ = count_fp_at(conf_min)
        total_tp += tp0
        total_fp += fp0
        fp_per_image.append(fp0)

        for t in thresholds:
            _, fpt, _ = count_fp_at(t)
            fp_by_thr[str(t)].append(fpt)

        if n_images == 1 or (n_images % 10 == 0):
            print(f"[B] {img_dir.name}: processed {n_images} images", flush=True)

    fp_arr = np.asarray(fp_per_image, dtype=np.float32)

    return {
        "images_eval": int(n_images),
        "gt_total_boxes": int(total_gt),
        "pred_total_boxes_conf_ge_conf_min": int(total_preds),
        "tp_total_conf_ge_conf_min": int(total_tp),
        "fp_total_conf_ge_conf_min": int(total_fp),
        "fp_per_image_mean_conf_min": float(fp_arr.mean()) if n_images else None,
        "fp_per_image_median_conf_min": float(np.median(fp_arr)) if n_images else None,
        "fp_per_image_p95_conf_min": percentile(fp_arr, 95),
        "fp_per_image_conf_ge": {str(t): float(np.mean(fp_by_thr[str(t)])) if n_images else None for t in thresholds},
        "config": {
            "img_dir_used": str(img_dir),
            "labels_dir_used": str(labels_dir),
            "imgsz": imgsz,
            "conf_min": conf_min,
            "iou": iou,
            "thresholds": thresholds,
            "gt_iou_match": gt_iou_match,
            "device": str(device_pred),
        },
    }

def delta_numeric(a, b):
    if isinstance(a, (int, float)) and isinstance(b, (int, float)):
        return b - a
    return None

def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--out_root", default="/home/nvidia/nova_edge_eval/yolo_eval_runs")
    ap.add_argument("--run_name", default=None)
    ap.add_argument("--model", required=True)

    # ✅ NEW: device selection (cpu, 0, 1, 0,1, cuda:0, mps)
    ap.add_argument("--device", default="cpu", help="Ultralytics device (e.g. cpu, 0, 1, 0,1, cuda:0)")

    # single root per dataset
    ap.add_argument("--clean_root", required=True, help="Root containing images/<split>/billboardXX and labels/<split>/billboardXX")
    ap.add_argument("--patched_root", required=True, help="Root containing images/<split>/billboardXX and labels/<split>/billboardXX")

    # choose split
    ap.add_argument("--split", default="test", choices=["train", "val", "test"])

    ap.add_argument("--imgsz", type=int, default=672)
    ap.add_argument("--conf_min", type=float, default=0.001)
    ap.add_argument("--iou", type=float, default=0.6)
    ap.add_argument("--thresholds", default="0.3,0.5,0.7")
    ap.add_argument("--gt_iou_match", type=float, default=0.5)

    ap.add_argument("--scenarios", default="1,2,3,4,5,6,7,8,9")
    ap.add_argument("--det_k", default="1,5,10,20,50")
    args = ap.parse_args()

    device_pred, device_cli = _normalize_device_arg(args.device)

    thresholds = [float(x.strip()) for x in args.thresholds.split(",") if x.strip()]
    det_k_list = [int(x.strip()) for x in args.det_k.split(",") if x.strip()]

    out_root = Path(args.out_root)
    out_root.mkdir(parents=True, exist_ok=True)
    if args.run_name:
        run_dir = out_root / args.run_name
    else:
        from datetime import datetime
        ts = datetime.now().strftime("%Y%m%d_%H%M%S")
        run_dir = out_root / f"run_{ts}"
    safe_mkdir(run_dir)

    yamls_dir = safe_mkdir(run_dir / "yamls")

    print(f"RUN DIR: {run_dir}", flush=True)
    print(f"MODEL: {args.model}", flush=True)
    print(f"DEVICE: {args.device}", flush=True)
    print(f"SPLIT: {args.split}", flush=True)
    print(f"IMG_SIZE: {args.imgsz}", flush=True)
    print(f"CLEAN_ROOT: {args.clean_root}", flush=True)
    print(f"PATCHED_ROOT: {args.patched_root}", flush=True)

    model = YOLO(args.model)
    names = model_names_as_list(model)

    scenarios = [f"billboard{int(s):02d}" for s in args.scenarios.split(",") if s.strip()]

    per_scenario = []
    clean_map50s = []; patched_map50s = []; delta_map50s = []
    clean_map5095s = []; patched_map5095s = []; delta_map5095s = []
    clean_det_means = []; patched_det_means = []; delta_det_means = []
    clean_fp_means = []; patched_fp_means = []; delta_fp_means = []

    for bb in scenarios:
        sc_dir = safe_mkdir(run_dir / bb)

        clean_img_dir = Path(args.clean_root) / "images" / args.split / bb
        clean_lab_dir = Path(args.clean_root) / "labels" / args.split / bb
        if not clean_img_dir.exists():
            raise FileNotFoundError(f"[{bb}] Missing clean images: {clean_img_dir}")
        if not clean_lab_dir.exists():
            raise FileNotFoundError(f"[{bb}] Missing clean labels: {clean_lab_dir}")

        patched_img_dir = Path(args.patched_root) / "images" / args.split / bb
        patched_lab_dir = Path(args.patched_root) / "labels" / args.split / bb
        if not patched_img_dir.exists():
            raise FileNotFoundError(f"[{bb}] Missing patched images: {patched_img_dir}")
        if not patched_lab_dir.exists():
            raise FileNotFoundError(f"[{bb}] Missing patched labels: {patched_lab_dir}")

        clean_yaml = yamls_dir / f"{bb}_clean_{args.split}.yaml"
        patched_yaml = yamls_dir / f"{bb}_patched_{args.split}.yaml"
        write_yaml(clean_yaml, Path(args.clean_root), args.split, bb, names)
        write_yaml(patched_yaml, Path(args.patched_root), args.split, bb, names)

        clean_val_log = sc_dir / f"{bb}_clean_val_{args.split}.log"
        patched_val_log = sc_dir / f"{bb}_patched_val_{args.split}.log"

        cmd_base = [
            "yolo", "val",
            f"model={args.model}",
            f"imgsz={args.imgsz}",
            f"conf={args.conf_min}",
            f"iou={args.iou}",
            f"split={args.split}",
            f"device={device_cli}",   # ✅ GPU/CPU for yolo val
            "batch=1",
            "workers=0",
            f"project={sc_dir}",
        ]

        rc = run(cmd_base + [f"data={clean_yaml}", f"name={bb}_clean_{args.split}"], clean_val_log)
        if rc != 0:
            raise SystemExit(f"yolo val clean failed for {bb} (rc={rc}) see {clean_val_log}")

        rc = run(cmd_base + [f"data={patched_yaml}", f"name={bb}_patched_{args.split}"], patched_val_log)
        if rc != 0:
            raise SystemExit(f"yolo val patched failed for {bb} (rc={rc}) see {patched_val_log}")

        clean_val = parse_yolo_val_metrics_from_log(clean_val_log)
        patched_val = parse_yolo_val_metrics_from_log(patched_val_log)

        clean_A = compute_detection_density_metrics(model, clean_img_dir, args.imgsz, args.conf_min, args.iou, thresholds, det_k_list, device_pred)
        patched_A = compute_detection_density_metrics(model, patched_img_dir, args.imgsz, args.conf_min, args.iou, thresholds, det_k_list, device_pred)

        clean_B = compute_false_positive_metrics(model, clean_img_dir, clean_lab_dir, args.imgsz, args.conf_min, args.iou, thresholds, args.gt_iou_match, device_pred)
        patched_B = compute_false_positive_metrics(model, patched_img_dir, patched_lab_dir, args.imgsz, args.conf_min, args.iou, thresholds, args.gt_iou_match, device_pred)

        delta = {
            "yolo_val": {
                "precision": delta_numeric(clean_val.get("precision"), patched_val.get("precision")),
                "recall": delta_numeric(clean_val.get("recall"), patched_val.get("recall")),
                "mAP50": delta_numeric(clean_val.get("mAP50"), patched_val.get("mAP50")),
                "mAP50-95": delta_numeric(clean_val.get("mAP50-95"), patched_val.get("mAP50-95")),
            },
            "A": {
                "detections_per_image_mean": delta_numeric(clean_A.get("detections_per_image_mean"), patched_A.get("detections_per_image_mean")),
                "detections_total_model": delta_numeric(clean_A.get("detections_total_model"), patched_A.get("detections_total_model")),
            },
            "B": {
                "fp_per_image_mean_conf_min": delta_numeric(clean_B.get("fp_per_image_mean_conf_min"), patched_B.get("fp_per_image_mean_conf_min")),
                "fp_total_conf_ge_conf_min": delta_numeric(clean_B.get("fp_total_conf_ge_conf_min"), patched_B.get("fp_total_conf_ge_conf_min")),
            },
        }

        entry = {
            "scenario": bb,
            "split": args.split,
            "model": args.model,
            "imgsz": args.imgsz,
            "device": args.device,
            "clean": {
                "root": str(args.clean_root),
                "img_dir": str(clean_img_dir),
                "labels_dir": str(clean_lab_dir),
                "yaml": str(clean_yaml),
                "yolo_val": clean_val,
                "A": clean_A,
                "B": clean_B,
            },
            "patched": {
                "root": str(args.patched_root),
                "img_dir": str(patched_img_dir),
                "labels_dir": str(patched_lab_dir),
                "yaml": str(patched_yaml),
                "yolo_val": patched_val,
                "A": patched_A,
                "B": patched_B,
            },
            "delta_patched_minus_clean": delta,
        }
        per_scenario.append(entry)
        (sc_dir / "metrics.json").write_text(json.dumps(entry, indent=2), encoding="utf-8")

        cm50 = clean_val.get("mAP50"); pm50 = patched_val.get("mAP50")
        cm95 = clean_val.get("mAP50-95"); pm95 = patched_val.get("mAP50-95")
        clean_map50s.append(cm50); patched_map50s.append(pm50); delta_map50s.append(delta_numeric(cm50, pm50))
        clean_map5095s.append(cm95); patched_map5095s.append(pm95); delta_map5095s.append(delta_numeric(cm95, pm95))

        cdet = clean_A.get("detections_per_image_mean"); pdet = patched_A.get("detections_per_image_mean")
        clean_det_means.append(cdet); patched_det_means.append(pdet); delta_det_means.append(delta_numeric(cdet, pdet))

        cfp = clean_B.get("fp_per_image_mean_conf_min"); pfp = patched_B.get("fp_per_image_mean_conf_min")
        clean_fp_means.append(cfp); patched_fp_means.append(pfp); delta_fp_means.append(delta_numeric(cfp, pfp))

    summary = {
        "run_dir": str(run_dir),
        "model": args.model,
        "imgsz": args.imgsz,
        "split": args.split,
        "device": args.device,
        "scenarios": scenarios,
        "mean_std": {
            "clean_mAP50": mean_std(clean_map50s),
            "patched_mAP50": mean_std(patched_map50s),
            "delta_mAP50": mean_std(delta_map50s),
            "clean_mAP50_95": mean_std(clean_map5095s),
            "patched_mAP50_95": mean_std(patched_map5095s),
            "delta_mAP50_95": mean_std(delta_map5095s),
            "clean_det_mean": mean_std(clean_det_means),
            "patched_det_mean": mean_std(patched_det_means),
            "delta_det_mean": mean_std(delta_det_means),
            "clean_fp_mean": mean_std(clean_fp_means),
            "patched_fp_mean": mean_std(patched_fp_means),
            "delta_fp_mean": mean_std(delta_fp_means),
        },
    }

    combined = {**summary, "per_scenario": per_scenario}

    out_combined = run_dir / "combined_metrics.json"
    out_summary = run_dir / "summary.json"
    out_combined.write_text(json.dumps(combined, indent=2), encoding="utf-8")
    out_summary.write_text(json.dumps(summary, indent=2), encoding="utf-8")

    print("\nWROTE:", out_combined, flush=True)
    print("WROTE:", out_summary, flush=True)
    print("\nSUMMARY:\n", json.dumps(summary, indent=2), flush=True)

if __name__ == "__main__":
    main()

In [ ]:
import shutil
from pathlib import Path

src_root = Path("/content/drive/MyDrive/Patched Datasets/patched_dataset1")
flat_img = src_root / "images" / "test"
flat_lab = src_root / "labels" / "test"

# Create billboard01 subfolders
nested_img = flat_img / "billboard01"
nested_lab = flat_lab / "billboard01"
nested_img.mkdir(exist_ok=True)
nested_lab.mkdir(exist_ok=True)

# Move images in
moved_imgs = 0
for f in list(flat_img.glob("*.png")) + list(flat_img.glob("*.jpg")):
    shutil.copy(str(f), str(nested_img / f.name))  # copy not move, keep original
    moved_imgs += 1

# Move labels in
moved_labs = 0
for f in flat_lab.glob("*.txt"):
    shutil.copy(str(f), str(nested_lab / f.name))
    moved_labs += 1

print(f"✅ Copied {moved_imgs} images and {moved_labs} labels into billboard01/")

In [ ]:
src_root = Path("/content/drive/MyDrive/Patched Datasets/Defended/patched_dataset1_defended_PatchBlock_SVD_info65")
flat_img = src_root / "images" / "test"
flat_lab = src_root / "labels" / "test"

nested_img = flat_img / "billboard01"
nested_lab = flat_lab / "billboard01"
nested_img.mkdir(exist_ok=True)
nested_lab.mkdir(exist_ok=True)

for f in list(flat_img.glob("*.png")) + list(flat_img.glob("*.jpg")):
    shutil.copy(str(f), str(nested_img / f.name))

for f in flat_lab.glob("*.txt"):
    shutil.copy(str(f), str(nested_lab / f.name))

print("✅ PatchBlock defended dataset restructured")

In [ ]:
import subprocess

result = subprocess.run([
    "python", "/content/evaluation_ab.py",
    "--model",        "/content/drive/MyDrive/yolo_trained_models/2dod_checkpoints/best.pt",
    "--clean_root",   "/content/drive/MyDrive/Patched Datasets/patched_dataset1",  # has billboard01 subfolders
    "--patched_root", "/content/drive/MyDrive/Patched Datasets/Defended/patched_dataset1_defended_PatchBlock_SVD_info65",
    "--run_name",     "PatchBlock_SVD_info65_eval",
    "--out_root",     "/content/drive/MyDrive/paper_runs",
    "--split",        "test",
    "--scenarios",    "1",          # only billboard01
    "--imgsz",        "640",
    "--conf_min",     "0.001",
    "--iou",          "0.6",
    "--thresholds",   "0.3,0.5,0.7",
    "--gt_iou_match", "0.5",
    "--device",       "0",
], capture_output=True, text=True)

print(result.stdout[-3000:])
print(result.stderr[-500:])

## 10. LGS vs PatchBlock Latency Comparison

In [ ]:
# Time LGS on same images
lgs_latencies = []
for p in tqdm(img_paths[:min(30, len(img_paths))], desc="Timing LGS"):
    img = np.array(Image.open(p).convert("RGB"))
    t0 = time.perf_counter()
    lgs_defense(img, sigma=LGS_SIGMA_CARLA, threshold=LGS_THR_CARLA)
    lgs_latencies.append((time.perf_counter() - t0) * 1000)

pb_latencies_sub = latencies[:len(lgs_latencies)]

print("=" * 45)
print(f"{'Defence':<20} {'Mean (ms)':>10} {'p95 (ms)':>10}")
print("=" * 45)
print(f"{'LGS':<20} {np.mean(lgs_latencies):>10.1f} {np.percentile(lgs_latencies,95):>10.1f}")
print(f"{'PatchBlock SVD':<20} {np.mean(pb_latencies_sub):>10.1f} {np.percentile(pb_latencies_sub,95):>10.1f}")
print("=" * 45)
print(f"Speedup (LGS vs PatchBlock): {np.mean(pb_latencies_sub)/np.mean(lgs_latencies):.1f}×")

# Bar chart
fig, ax = plt.subplots(figsize=(6, 4))
methods = ['LGS', 'PatchBlock\n(SVD, info=0.65)']
means   = [np.mean(lgs_latencies), np.mean(pb_latencies_sub)]
stds    = [np.std(lgs_latencies),  np.std(pb_latencies_sub)]
colors  = ['#55A868', '#DD8452']
bars = ax.bar(methods, means, yerr=stds, capsize=6, color=colors, width=0.5)
ax.set_ylabel("Latency per image (ms)")
ax.set_title("Defence Pre-processing Latency")
for bar, m in zip(bars, means):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + max(stds)*0.15,
            f"{m:.0f} ms", ha='center', va='bottom', fontweight='bold')
plt.tight_layout()
save_fig("defence_latency_comparison")
plt.show()

## 11. YOLO Detection Comparison — Patched vs LGS vs PatchBlock

In [ ]:
model.model.eval()

def yolo_predict_img(img_rgb, conf=0.3):
    t = torch.from_numpy(img_rgb).permute(2,0,1).float().to(device) / 255.0
    with torch.no_grad():
        result = model(t.unsqueeze(0), verbose=False)[0]
    vis = cv2.cvtColor(result.plot(), cv2.COLOR_BGR2RGB)
    n   = 0 if result.boxes is None else len(result.boxes)
    return vis, n

vis_patched, n_p = yolo_predict_img(patched)
vis_lgs,     n_l = yolo_predict_img(defended_lgs)
vis_pb,      n_b = yolo_predict_img(defended_pb)

fig, axes = plt.subplots(1, 3, figsize=(21, 6))
axes[0].imshow(vis_patched); axes[0].set_title(f"Patched — {n_p} detections")
axes[1].imshow(vis_lgs);     axes[1].set_title(f"LGS — {n_l} detections")
axes[2].imshow(vis_pb);      axes[2].set_title(f"PatchBlock SVD — {n_b} detections")
for ax in axes: ax.axis("off")
plt.suptitle("YOLOv8n Detections: No Defence / LGS / PatchBlock", fontsize=13, y=1.01)
plt.tight_layout()
save_fig("yolo_detection_comparison_defences")
plt.show()

print(f"Detection counts:  Patched={n_p}  |  LGS={n_l}  |  PatchBlock={n_b}")

## 12. PatchBlock Latency Real-Time Assessment Summary

In [ ]:
DETECTOR_FPS     = 1000 / 45.2   # YOLOv8n full-pipeline mean E2E from thesis
DETECTOR_MS      = 45.2
PB_MEAN_MS       = np.mean(latencies)
LGS_MEAN_MS      = np.mean(lgs_latencies)
COMBINED_PB_MS   = DETECTOR_MS + PB_MEAN_MS
COMBINED_LGS_MS  = DETECTOR_MS + LGS_MEAN_MS

print("=" * 60)
print("REAL-TIME FEASIBILITY ASSESSMENT")
print("=" * 60)
print(f"Target: 30 FPS = {1000/30:.1f} ms/frame budget")
print()
print(f"YOLOv8n full pipeline (no defence): {DETECTOR_MS:.1f} ms  ({1000/DETECTOR_MS:.1f} FPS)")
print()
print(f"LGS overhead:       {LGS_MEAN_MS:.1f} ms")
print(f"  Combined E2E:     {COMBINED_LGS_MS:.1f} ms  ({1000/COMBINED_LGS_MS:.1f} FPS)  ✅ REAL-TIME")
print()
print(f"PatchBlock overhead:{PB_MEAN_MS:.1f} ms")
print(f"  Combined E2E:     {COMBINED_PB_MS:.1f} ms  ({1000/COMBINED_PB_MS:.1f} FPS)  ❌ NOT REAL-TIME")
print()
print(f"PatchBlock is {COMBINED_PB_MS/COMBINED_LGS_MS:.1f}× slower than LGS as a preprocessing step.")
print("=" * 60)

# Save summary
summary = {
    "Defence": ["None", "LGS", "PatchBlock SVD"],
    "Defence latency (ms)": [0, round(LGS_MEAN_MS,1), round(PB_MEAN_MS,1)],
    "Total E2E (ms)": [DETECTOR_MS, round(COMBINED_LGS_MS,1), round(COMBINED_PB_MS,1)],
    "FPS": [round(1000/DETECTOR_MS,1), round(1000/COMBINED_LGS_MS,1), round(1000/COMBINED_PB_MS,1)],
    "Real-time (≥30 FPS)": ["✅", "✅", "❌"]
}
pd.DataFrame(summary).to_csv(THESIS_FIGS / "realtime_feasibility_table.csv", index=False)
print("\n✅ Saved feasibility table.")

## 13. Summary of All Saved Thesis Figures

In [ ]:
saved = sorted(THESIS_FIGS.glob("*"))
print(f"\n{'='*55}")
print(f"THESIS FIGURES SAVED TO: {THESIS_FIGS}")
print(f"{'='*55}")
for f in saved:
    size_kb = f.stat().st_size / 1024
    print(f"  {f.name:<50} {size_kb:>7.1f} KB")
print(f"{'='*55}")
print(f"Total: {len(saved)} files")

# Zip everything for easy download
shutil.make_archive("/content/thesis_figures", "zip", THESIS_FIGS)
print("\n✅ All figures zipped to /content/thesis_figures.zip")
print("   Download via Files panel on the left.")